# Cost, Latency, and Agent Economics

**Level:** Advanced · **Time:** 60 min

Running flagship reasoning models for every user request will bankrupt your project. In production, we optimize for latency and cost using Caching and Model Routing.

In this notebook, we will simulate four critical optimization patterns:
1. **Semantic Caching:** Using mocked vector cosine similarity to bypass LLM calls for phrased variations of the same intent.
2. **Intent Classification (Small Model Front-Door):** Routing casual greetings away from the reasoning agent.
3. **LLM Cascades (FrugalGPT Pattern):** Routing a request to a cheap model, detecting a validation failure, and automatically rescuing it with an expensive model.
4. **Parallel Tool Execution:** Demonstrating the latency difference between sequential and parallel API calls.

---
## Pattern 1: Semantic Caching

Traditional caching requires exact string matches (`"hello"` != `"hi"`). Semantic caching uses Vector Embeddings to measure intent distance (Cosine Similarity). If the similarity is above a threshold, we return a cached response.

In [ ]:
import numpy as np

# A very simplified mock of a Vector Database storing Embeddings
# In reality, you'd use text-embedding-3-small and Pinecone/pgvector.
mock_vector_db = {
    "intent_reset_password": {
        "vector": np.array([0.9, 0.1, 0.0]),
        "cached_response": "To reset your password, visit northstar.internal/reset"
    }
}

# Mock embedding function
def mock_embed(text: str) -> np.ndarray:
    if "password" in text.lower() and "reset" in text.lower():
        return np.array([0.88, 0.12, 0.0]) # Very similar to intent_reset_password
    elif "password" in text.lower() and "forgot" in text.lower():
        return np.array([0.85, 0.15, 0.0]) # Still very similar
    else:
        return np.array([0.1, 0.9, 0.0]) # Completely different

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def check_semantic_cache(user_query: str) -> str:
    print(f"\n[User] Query: '{user_query}'")
    query_vector = mock_embed(user_query)
    
    # Check against DB
    for intent, data in mock_vector_db.items():
        similarity = cosine_similarity(query_vector, data["vector"])
        print(f"[Semantic Cache] Evaluated against {intent}. Similarity: {similarity:.2f}")
        
        if similarity > 0.95:
            print(f"[Semantic Cache] CACHE HIT! Cost: $0.00. Latency: 50ms.")
            return data["cached_response"]
            
    print("[Semantic Cache] CACHE MISS. Routing to LLM (Cost: $$$).")
    return "LLM_GENERATED_RESPONSE"

# 1. Exact match intent
check_semantic_cache("How do I reset my password?")

# 2. Phrased differently, but semantically identical
check_semantic_cache("I forgot my password, help!")

# 3. Completely different intent
check_semantic_cache("What is the weather in London?")



[User] Query: 'How do I reset my password?'
[Semantic Cache] Evaluated against intent_reset_password. Similarity: 1.00
[Semantic Cache] CACHE HIT! Cost: $0.00. Latency: 50ms.

[User] Query: 'I forgot my password, help!'
[Semantic Cache] Evaluated against intent_reset_password. Similarity: 0.99
[Semantic Cache] CACHE HIT! Cost: $0.00. Latency: 50ms.

[User] Query: 'What is the weather in London?'
[Semantic Cache] Evaluated against intent_reset_password. Similarity: 0.22
[Semantic Cache] CACHE MISS. Routing to LLM (Cost: $$$).


---
## Pattern 2: Intent Classification (Small Model Front-Door)

You do not need a flagship model to reply to a casual greeting. A small model (or semantic router) can intercept simple queries.

In [ ]:
def fast_intent_classifier(query: str):
    print(f"\n[Intent Classifier: Llama-3-8B] Processing: '{query}'")
    
    if "hello" in query.lower() or "hi" in query.lower():
        print("[Router] Intent: GREETING. Returning pre-canned response.")
        return "Hello! How can I help you today?"
        
    print("[Router] Intent: COMPLEX. Routing to Reasoning Agent (gpt-4o).")
    return "ROUTE_TO_REASONING"

fast_intent_classifier("Hello there!")
fast_intent_classifier("Can you debug this python script for me?")



[Intent Classifier: Llama-3-8B] Processing: 'Hello there!'
[Router] Intent: GREETING. Returning pre-canned response.

[Intent Classifier: Llama-3-8B] Processing: 'Can you debug this python script for me?'
[Router] Intent: COMPLEX. Routing to Reasoning Agent (gpt-4o).


---
## Pattern 3: LLM Cascades (FrugalGPT)

We ask a very cheap model (`gpt-4o-mini`) to extract data. It hallucinates and misses a required field. The orchestrator catches the `ValidationError` and falls back to the expensive model (`gpt-4o`) to rescue the workflow.

In [ ]:
def mock_gpt_4o_mini(prompt: str):
    # The cheap model hallucinates and forgets the 'amount' field
    print("[LLM: gpt-4o-mini] Cost: $0.0001")
    return {"action": "refund", "currency": "USD"} # Missing 'amount'

def mock_gpt_4o(prompt: str, error_context: str = None):
    # The expensive model gets it right
    print(f"[LLM: gpt-4o] Cost: $0.0050")
    if error_context:
        print(f"  > Fixing error: {error_context}")
    return {"action": "refund", "currency": "USD", "amount": 50.00}

def execute_cascade(prompt: str):
    print(f"\n[Orchestrator] Task: {prompt}")
    
    try:
        # Attempt 1: Fast & Cheap
        print("[Orchestrator] Attempt 1: Routing to Cheap Model...")
        result = mock_gpt_4o_mini(prompt)
        
        # Pydantic Validation Check
        if "amount" not in result:
            raise ValueError("Pydantic ValidationError: Field 'amount' is required.")
            
        print("[Orchestrator] Success!")
        return result
        
    except ValueError as e:
        print(f"[Orchestrator] Validation Failed: {e}")
        # Attempt 2: Rescue with Expensive Model
        print("[Orchestrator] Attempt 2: FALLBACK to Expensive Model...")
        result = mock_gpt_4o(prompt, error_context=str(e))
        
        print("[Orchestrator] Rescue Success!")
        return result

# Execute the workflow
final_output = execute_cascade("Refund the user $50.")
print(f"\n[Final Result] {final_output}")



[Orchestrator] Task: Refund the user $50.
[Orchestrator] Attempt 1: Routing to Cheap Model...
[LLM: gpt-4o-mini] Cost: $0.0001
[Orchestrator] Validation Failed: Pydantic ValidationError: Field 'amount' is required.
[Orchestrator] Attempt 2: FALLBACK to Expensive Model...
[LLM: gpt-4o] Cost: $0.0050
  > Fixing error: Pydantic ValidationError: Field 'amount' is required.
[Orchestrator] Rescue Success!

[Final Result] {'action': 'refund', 'currency': 'USD', 'amount': 50.0}


---
## Pattern 4: Parallel Tool Execution

If an agent needs to retrieve information from three different APIs, doing so sequentially is a massive latency hit. Modern orchestrators execute read-only tools concurrently.

In [ ]:
import time
import asyncio

async def fetch_api(source: str):
    print(f"  [API] Fetching data from {source}...")
    await asyncio.sleep(2) # Simulate 2-second network latency
    return f"{source} Data"

async def sequential_execution():
    print("\n[Orchestrator] Starting SEQUENTIAL Execution...")
    start_time = time.time()
    
    res1 = await fetch_api("Weather")
    res2 = await fetch_api("News")
    res3 = await fetch_api("Stocks")
    
    elapsed = time.time() - start_time
    print(f"[Orchestrator] Sequential finished in {elapsed:.2f} seconds.")

async def parallel_execution():
    print("\n[Orchestrator] Starting PARALLEL Execution...")
    start_time = time.time()
    
    # Execute all three simultaneously
    results = await asyncio.gather(
        fetch_api("Weather"),
        fetch_api("News"),
        fetch_api("Stocks")
    )
    
    elapsed = time.time() - start_time
    print(f"[Orchestrator] Parallel finished in {elapsed:.2f} seconds.")

# To run asyncio in a Jupyter notebook without crashing the existing loop:
await sequential_execution()
await parallel_execution()



[Orchestrator] Starting SEQUENTIAL Execution...
  [API] Fetching data from Weather...
  [API] Fetching data from News...
  [API] Fetching data from Stocks...
[Orchestrator] Sequential finished in 6.01 seconds.

[Orchestrator] Starting PARALLEL Execution...
  [API] Fetching data from Weather...
  [API] Fetching data from News...
  [API] Fetching data from Stocks...
[Orchestrator] Parallel finished in 2.00 seconds.
